In [5]:
import folium
import json
import numpy as np
from scipy.spatial import Delaunay
from folium.plugins import FloatImage
import os

# Définir le chemin du fichier dans le dossier parent
file_path = os.path.abspath(os.path.join(os.getcwd(), "..", "velib_stations.json"))

# Charger les données depuis le fichier JSON
with open(file_path, "r", encoding="utf-8") as file:
    stations = json.load(file)["data"]["stations"]

# Extraire les coordonnées (latitude, longitude)
points = np.array([[s["lat"], s["lon"]] for s in stations])

# Effectuer la Triangulation de Delaunay
Triangulation = Delaunay(points)

# Centrer la carte sur Paris
paris_coords = (48.8566, 2.3522)
m = folium.Map(location=paris_coords, zoom_start=12)

# Ajouter les stations Vélib' sur la carte
for s in stations:
    folium.Marker(
        location=[s["lat"], s["lon"]],
        popup=s["name"],
        icon=folium.Icon(color="blue", icon="info-sign")
    ).add_to(m)

# Ajouter les Triangles de Delaunay sous forme de lignes
for simplex in Triangulation.simplices:
    point = [points[i] for i in simplex]  # Récupérer les sommets du Triangle
    folium.PolyLine(locations=point + [point[0]], color="red", weight=2).add_to(m)  # Fermer le Triangle

# Ajouter une légende personnalisée
legend_html = """
<div style="
    position: fixed;
    bottom: 50px;
    right: 50px;
    background-color: white;
    padding: 10px;
    border-radius: 8px;
    box-shadow: 2px 2px 6px rgba(0,0,0,0.3);
    font-size: 14px;
    z-index: 999;
">
    <b>Légende :</b><br>
    🚲 <span style="color:blue;">Points Bleus</span> - Stations Vélib'<br>
    🔺 <span style="color:red;">Lignes Rouges</span> - Triangulation de Delaunay
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m.save("velib_delaunay_legend.html")

In [6]:
import json
import numpy as np
from scipy.spatial import Delaunay
import os

# Définir le chemin du fichier dans le dossier parent
file_path = os.path.abspath(os.path.join(os.getcwd(), "..", "velib_stations.json"))

# Charger les données depuis le fichier JSON
with open(file_path, "r", encoding="utf-8") as file:
    stations = json.load(file)["data"]["stations"]

# Extraire les noms des stations et leurs coordonnées (lat, lon)
nom_station = [s["name"] for s in stations]  # Liste des noms
coo = np.array([[s["lat"], s["lon"]] for s in stations])  # Coordonnées

# Effectuer la Tri de Delaunay
Triangulation = Delaunay(coo)

# Initialiser la liste d'adjacence
Liste_adjacence = {name: set() for name in nom_station}  # Dictionnaire de sets

# Remplir la liste d'adjacence à partir des Triangles
for simplex in Triangulation.simplices:  # simplex = un Triangle (3 sommets)
    for i in range(3):  # Parcourir les 3 sommets du Triangle
        for j in range(3):
            if i != j:  # Ne pas ajouter une station comme voisine d'elle-même
                station_i = nom_station[simplex[i]]
                station_j = nom_station[simplex[j]]
                Liste_adjacence[station_i].add(station_j)

# Convertir les sets en listes pour un affichage JSON-friendly
Liste_adjacence = {k: list(v) for k, v in Liste_adjacence.items()}

# Afficher un extrait de la liste d'adjacence
print(json.dumps(Liste_adjacence, indent=4, ensure_ascii=False))

# Sauvegarder la liste d'adjacence dans un fichier JSON
with open("velib_Liste_adjacence.json", "w", encoding="utf-8") as f:
    json.dump(Liste_adjacence, f, indent=4, ensure_ascii=False)


{
    "Benjamin Godard - Victor Hugo": [
        "Mairie du 16ème",
        "Victor Hugo - La Pompe",
        "Flandrin - Longchamp",
        "Flandrin - Henri Martin"
    ],
    "Hôpital Mondor": [
        "Bleuets - Bordières",
        "Créteil Village",
        "Les Juilliottes",
        "Université Paris Est Créteil",
        "Préfecture de Créteil",
        "Centre Hospitalier Intercommunal de Créteil"
    ],
    "Rouget de L'isle - Watteau": [
        "Camille Risch - Paul Armangot",
        "Conservatoire de Musique",
        "Youri Gagarine - Commune de Paris",
        "Balzac - Olympes de Gouges",
        "8 Mai 1945 - 10 Juillet 1940",
        "Lebrun - Colonel Fabien "
    ],
    "Toudouze - Clauzel": [
        "Victor Massé - Jean-Baptiste Pigalle",
        "Jean-Baptiste Pigalle - La Bruyere",
        "Lallier - Trudaine ",
        "Saint Georges - d'Aumale",
        "Choron - Martyrs"
    ],
    "Cassini - Denfert-Rochereau": [
        "Saint-Jacques - Tombe Issoire",
   